# Offline skill of the rho-flux ANN

Quantitative offline skill (R$^2$ and correlation) of the canonical buoyancy/density-flux ANN on the CM2.6 test set, across coarse-graining factors and depth. Buoyancy-flux counterpart of `notebooks/Figure-1.ipynb` (momentum), reusing `predict_ANN_rho` + `SGS_skill_rho`.

Run top-to-bottom on a machine with the `/vast` CM2.6 test data and the JAX/torch env. The compute cell is slow (~minutes per factor).

In [1]:
import sys
sys.path.append('../src/training-on-CM2.6')

import os, gc
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cmocean

from helpers.cm26 import read_datasets
from helpers.ann_tools import import_ANN

## Config

In [ ]:
factors = [4, 9, 12, 15]
stencil_size = 3

# Canonical buoyancy ANN (committed in-repo: stencil 3, hidden [32,32], FGR3, EXP0)
ann_path = '../CM26_ML_models/ocean3d/subfilter/FGR3/buoyancy/hidden-layer-32-32/seed-default/model/ann_instance_20Dec.nc'

# Cache per-factor skill datasets so plotting does not trigger recompute.
skill_dir = os.path.expandvars('/scratch/$USER/mom6/CM26_ML_models/FGR3/EXP0/skill-test-rho')
os.makedirs(skill_dir, exist_ok=True)

## Compute skill on the test set (slow: ~minutes per factor)

Loops `ANN_rho_inference` over time/depth via `predict_ANN_rho`, then computes R$^2$F / corr_F via `SGS_skill_rho`. Saves one file per factor.

In [ ]:
ann = import_ANN(ann_path)
ds = read_datasets(['test'], factors)

for factor in factors:
    skill = ds[f'test-{factor}'].predict_ANN_rho(ann, stencil_size=stencil_size).SGS_skill_rho()
    skill.to_netcdf(f'{skill_dir}/factor-{factor}.nc')
    print(f'factor {factor}:  R2F={float(skill.R2F.mean()):.3f}  corr_F={float(skill.corr_F.mean()):.3f}')
    del skill; gc.collect()

## R$^2$ and correlation across grid spacing and depth

In [ ]:
def read_skill(skill_dir, factors):
    return {f: xr.open_dataset(f'{skill_dir}/factor-{f}.nc') for f in factors}

def metric_array(skill, metric):
    data = xr.concat([skill[f][metric] for f in factors], dim='factor')
    return data.assign_coords(factor=factors)

skill = read_skill(skill_dir, factors)

In [ ]:
# Factor x depth heatmaps
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, vmin in zip(axes, ['R2F', 'corr_F'], [-1, 0]):
    data = metric_array(skill, metric)
    data.plot(x='factor', y='zl', ax=ax, cmap=cmocean.cm.balance, vmin=vmin, vmax=1,
              cbar_kwargs={'label': metric})
    ax.invert_yaxis()
    ax.set_title(f'{metric}   mean={float(data.mean()):.2f}')
    ax.set_xlabel('coarse-graining factor'); ax.set_ylabel('depth (zl)')
plt.tight_layout()
plt.savefig('offline_skill_rho_heatmap.pdf', dpi=150)

In [ ]:
# Depth profiles, one line per factor
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ['R2F', 'corr_F']):
    data = metric_array(skill, metric)
    for f in factors:
        ax.plot(data.sel(factor=f), data.zl, label=f'factor {f}')
    ax.invert_yaxis(); ax.set_xlim(right=1)
    ax.set_xlabel(metric); ax.set_ylabel('depth (zl)')
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()

## Spatial check: true vs predicted flux

Surface snapshot of the meridional density flux (truth, ANN, error) at one factor.

In [ ]:
f = 9
sk = skill[f]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
panels = [('truth $F_y$', sk.Fy), ('ANN $F_y$', sk.Fy_pred), ('error', sk.Fy - sk.Fy_pred)]
for ax, (name, arr) in zip(axes, panels):
    arr.isel(zl=0).plot(ax=ax, robust=True, cmap=cmocean.cm.balance)
    ax.set_title(f'{name} (factor {f}, surface)')
plt.tight_layout()